In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Live_Query_Engine_Test") \
    .master("local[2]") \
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
            "org.apache.hadoop:hadoop-aws:3.4.1,"
            "software.amazon.awssdk:bundle:2.29.38") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local",           "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type",      "rest") \
    .config("spark.sql.catalog.local.uri",       "http://rest-catalog:8181") \
    .config("spark.sql.catalog.local.warehouse", "s3a://business-data/") \
    .config("spark.sql.catalog.local.s3.endpoint",          "http://minio:9000") \
    .config("spark.sql.catalog.local.s3.path-style-access", "true") \
    .config("spark.sql.catalog.local.s3.region",            "us-east-1") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "admin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Session Ready! Version: {spark.version}")

Spark Session Ready! Version: 4.0.2


In [4]:
# Query the live Iceberg table
spark.sql("""
    SELECT 
        traffic_event_ts, 
        traffic_borough, 
        traffic_speed, 
        aq_pm25_ugm3, 
        weather_temperature
    FROM local.db.enriched_traffic
    ORDER BY traffic_event_ts DESC
    LIMIT 10
""").show(truncate=False)

+-------------------+---------------+-------------+------------+-------------------+
|traffic_event_ts   |traffic_borough|traffic_speed|aq_pm25_ugm3|weather_temperature|
+-------------------+---------------+-------------+------------+-------------------+
|2026-05-07 02:08:12|Staten Island  |57.16        |NULL        |NULL               |
|2026-05-07 02:08:12|Staten Island  |60.89        |NULL        |NULL               |
|2026-05-07 02:08:12|Staten Island  |57.16        |NULL        |NULL               |
|2026-05-07 02:08:12|Staten Island  |59.65        |NULL        |NULL               |
|2026-05-07 02:08:12|Staten Island  |0.0          |NULL        |NULL               |
|2026-05-07 02:08:12|Queens         |25.47        |NULL        |NULL               |
|2026-05-07 02:08:12|Staten Island  |0.0          |NULL        |NULL               |
|2026-05-07 02:08:12|Queens         |31.68        |NULL        |NULL               |
|2026-05-07 02:08:12|Staten Island  |57.16        |NULL        |N

In [5]:
# Verify the aggregations and non-null air quality readings
spark.sql("""
    SELECT 
        traffic_borough, 
        COUNT(*) as total_records,
        ROUND(AVG(traffic_speed), 2) as avg_speed_mph, 
        ROUND(AVG(aq_pm25_ugm3), 2) as avg_pm25,
        MAX(weather_temperature) as current_temp
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough
    ORDER BY avg_speed_mph ASC
""").show(truncate=False)

+---------------+-------------+-------------+--------+------------+
|traffic_borough|total_records|avg_speed_mph|avg_pm25|current_temp|
+---------------+-------------+-------------+--------+------------+
|Manhattan      |413422       |16.55        |11.71   |66          |
|Staten Island  |40852        |28.68        |20.44   |66          |
|Bronx          |176168       |30.41        |10.95   |66          |
|Brooklyn       |110409       |30.73        |16.98   |66          |
|Queens         |132808       |38.23        |16.23   |66          |
+---------------+-------------+-------------+--------+------------+



In [6]:
# 1. Read the raw Parquet dumps directly from MinIO
raw_traffic = spark.read.parquet("s3a://raw-data/traffic/")
raw_traffic.createOrReplaceTempView("raw_traffic_view")

# 2. See how many records have successfully landed
spark.sql("""
    SELECT borough, COUNT(*) as rows_ingested 
    FROM raw_traffic_view 
    GROUP BY borough
""").show()

+-------------+-------------+
|      borough|rows_ingested|
+-------------+-------------+
|       Queens|         1887|
|     Brooklyn|          612|
|Staten Island|         1326|
|    Manhattan|         1326|
|        Bronx|         1224|
+-------------+-------------+



In [7]:
# Borough-level diagnostics: traffic-only vs traffic+AQ vs traffic+AQ+weather
spark.sql("""
WITH traffic_only AS (
  SELECT borough AS traffic_borough, COUNT(*) AS traffic_rows
  FROM parquet.`s3a://raw-data/traffic/`
  WHERE borough IS NOT NULL
  GROUP BY borough
),
aq_joined AS (
  SELECT traffic_borough, COUNT(*) AS traffic_aq_rows
  FROM local.db.enriched_traffic
  GROUP BY traffic_borough
),
aq_weather_joined AS (
  SELECT traffic_borough, COUNT(*) AS traffic_aq_weather_rows
  FROM local.db.enriched_traffic
  WHERE weather_temperature IS NOT NULL
  GROUP BY traffic_borough
)
SELECT
  t.traffic_borough,
  t.traffic_rows,
  COALESCE(a.traffic_aq_rows, 0) AS traffic_aq_rows,
  COALESCE(w.traffic_aq_weather_rows, 0) AS traffic_aq_weather_rows,
  ROUND(100.0 * COALESCE(a.traffic_aq_rows, 0) / t.traffic_rows, 2) AS aq_match_pct,
  ROUND(100.0 * COALESCE(w.traffic_aq_weather_rows, 0) / t.traffic_rows, 2) AS aq_weather_match_pct
FROM traffic_only t
LEFT JOIN aq_joined a ON t.traffic_borough = a.traffic_borough
LEFT JOIN aq_weather_joined w ON t.traffic_borough = w.traffic_borough
ORDER BY t.traffic_rows DESC
""").show(truncate=False)

+---------------+------------+---------------+-----------------------+------------+--------------------+
|traffic_borough|traffic_rows|traffic_aq_rows|traffic_aq_weather_rows|aq_match_pct|aq_weather_match_pct|
+---------------+------------+---------------+-----------------------+------------+--------------------+
|Queens         |1887        |132808         |132217                 |7038.05     |7006.73             |
|Staten Island  |1326        |40852          |40436                  |3080.84     |3049.47             |
|Manhattan      |1326        |413422         |413006                 |31178.13    |31146.76            |
|Bronx          |1224        |176168         |175784                 |14392.81    |14361.44            |
|Brooklyn       |612         |110409         |110233                 |18040.69    |18011.93            |
+---------------+------------+---------------+-----------------------+------------+--------------------+



In [8]:
spark.read.parquet("s3a://raw-data/weather/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

spark.read.parquet("s3a://raw-data/openaq/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-05-06 22:58:50.334|2026-05-07 04:33:17.558|
+-----------------------+-----------------------+

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-05-06 22:58:50.598|2026-05-07 04:33:44.126|
+-----------------------+-----------------------+



In [9]:
spark.sql("""
    SELECT MIN(traffic_event_ts), MAX(traffic_event_ts), COUNT(*) as total
    FROM local.db.enriched_traffic
""").show(truncate=False)

+---------------------+---------------------+------+
|min(traffic_event_ts)|max(traffic_event_ts)|total |
+---------------------+---------------------+------+
|2026-05-06 21:18:00  |2026-05-07 02:08:12  |873659|
+---------------------+---------------------+------+



In [10]:
spark.sql("""
    SELECT 
        MAX(traffic_event_ts) as latest_traffic,
        (SELECT MIN(kafka_timestamp) FROM parquet.`s3a://raw-data/openaq/`) as earliest_aq
    FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-----------------------+
|latest_traffic     |earliest_aq            |
+-------------------+-----------------------+
|2026-05-07 02:08:12|2026-05-06 22:58:50.598|
+-------------------+-----------------------+



In [11]:
spark.sql("""
    SELECT 
        traffic_borough,
        traffic_id,
        COUNT(*) as aq_matches_per_traffic_row
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough, traffic_id
    ORDER BY aq_matches_per_traffic_row DESC
    LIMIT 20
""").show(truncate=False)

+---------------+----------+--------------------------+
|traffic_borough|traffic_id|aq_matches_per_traffic_row|
+---------------+----------+--------------------------+
|Manhattan      |445       |64355                     |
|Brooklyn       |154       |32357                     |
|Manhattan      |149       |32357                     |
|Brooklyn       |148       |32357                     |
|Brooklyn       |157       |32357                     |
|Manhattan      |448       |32357                     |
|Manhattan      |106       |32357                     |
|Manhattan      |124       |32189                     |
|Manhattan      |221       |26968                     |
|Manhattan      |215       |26968                     |
|Manhattan      |364       |26828                     |
|Manhattan      |223       |21579                     |
|Manhattan      |217       |21579                     |
|Manhattan      |145       |21579                     |
|Manhattan      |150       |21579               

In [12]:
spark.sql("""
    SELECT 
        h3_index,
        COUNT(*) as rows
    FROM local.db.enriched_traffic
    GROUP BY h3_index
    ORDER BY rows DESC
    LIMIT 10
""").show(truncate=False)

+---------------+------+
|h3_index       |rows  |
+---------------+------+
|872a10729ffffff|161617|
|872a100aeffffff|145710|
|872a1072dffffff|86316 |
|872a100d2ffffff|80764 |
|872a1008bffffff|64714 |
|872a1072cffffff|64355 |
|872a100f2ffffff|53837 |
|872a10623ffffff|32472 |
|872a1001effffff|21648 |
|872a1000bffffff|21620 |
+---------------+------+



In [13]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic,
  MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq,
  MAX(aq_event_ts) AS max_aq,
  MIN(weather_event_ts) AS min_weather,
  MAX(weather_event_ts) AS max_weather
FROM local.db.enriched_traffic
""").show(truncate=False)

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_rows,
  SUM(CASE WHEN weather_temperature IS NOT NULL THEN 1 ELSE 0 END) AS weather_rows
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|min_traffic        |max_traffic        |min_aq             |max_aq             |min_weather        |max_weather        |
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|2026-05-06 21:18:00|2026-05-07 02:08:12|2026-05-06 22:58:50|2026-05-07 00:13:02|2026-05-06 22:47:04|2026-05-07 00:10:09|
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+

+----------+-------+------------+
|total_rows|aq_rows|weather_rows|
+----------+-------+------------+
|873659    |845065 |871676      |
+----------+-------+------------+



In [14]:
# AQ Debugging

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_matched_rows,
  ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS aq_match_pct
FROM local.db.enriched_traffic
""").show(truncate=False)

+----------+---------------+------------+
|total_rows|aq_matched_rows|aq_match_pct|
+----------+---------------+------------+
|873659    |845065         |96.73       |
+----------+---------------+------------+



In [15]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic, MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq, MAX(aq_event_ts) AS max_aq
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+
|min_traffic        |max_traffic        |min_aq             |max_aq             |
+-------------------+-------------------+-------------------+-------------------+
|2026-05-06 21:18:00|2026-05-07 02:08:12|2026-05-06 22:58:50|2026-05-07 00:13:02|
+-------------------+-------------------+-------------------+-------------------+



In [16]:
spark.sql("""
SELECT
  traffic_borough,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_non_null_rows,
  ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS aq_non_null_pct
FROM local.db.enriched_traffic
GROUP BY traffic_borough
ORDER BY traffic_borough;
""").show(truncate=False)

+---------------+----------+----------------+---------------+
|traffic_borough|total_rows|aq_non_null_rows|aq_non_null_pct|
+---------------+----------+----------------+---------------+
|Bronx          |176168    |172448          |97.89          |
|Brooklyn       |110409    |107780          |97.62          |
|Manhattan      |413422    |408864          |98.90          |
|Queens         |132808    |123639          |93.10          |
|Staten Island  |40852     |32334           |79.15          |
+---------------+----------+----------------+---------------+



In [17]:
#Speed bucket -> PM2.5 distribution (the core feedback loop)
spark.sql("""
    SELECT
        CASE
            WHEN traffic_speed < 10 THEN '0-10 mph (gridlock)'
            WHEN traffic_speed < 25 THEN '10-25 mph (slow)'
            WHEN traffic_speed < 45 THEN '25-45 mph (moderate)'
            ELSE '45+ mph (free flow)'
        END AS speed_bucket,
        COUNT(*) AS records,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25,
        ROUND(STDDEV(aq_pm25_ugm3), 2) AS stddev_pm25
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL
    GROUP BY 1
    ORDER BY avg_pm25 DESC
""").show(truncate=False)

+--------------------+-------+--------+-----------+
|speed_bucket        |records|avg_pm25|stddev_pm25|
+--------------------+-------+--------+-----------+
|45+ mph (free flow) |115413 |17.64   |9.3        |
|10-25 mph (slow)    |256079 |14.43   |9.6        |
|25-45 mph (moderate)|298253 |12.91   |9.61       |
|0-10 mph (gridlock) |175320 |9.1     |7.16       |
+--------------------+-------+--------+-----------+



In [18]:
# Congestion zone vs non-congestion zone PM2.5
spark.sql("""
    SELECT
        is_in_congestion_zone,
        is_near_truck_route,
        COUNT(*) AS records,
        ROUND(AVG(traffic_speed), 2) AS avg_speed,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL
    GROUP BY is_in_congestion_zone, is_near_truck_route
    ORDER BY avg_pm25 DESC
""").show(truncate=False)

+---------------------+-------------------+-------+---------+--------+
|is_in_congestion_zone|is_near_truck_route|records|avg_speed|avg_pm25|
+---------------------+-------------------+-------+---------+--------+
|false                |true               |672757 |26.91    |13.86   |
|false                |false              |172308 |17.11    |10.75   |
+---------------------+-------------------+-------+---------+--------+



In [19]:
#Hour-of-day pattern — rush hour signal
spark.sql("""
    SELECT
        HOUR(traffic_event_ts) AS hour_of_day,
        ROUND(AVG(traffic_speed), 2) AS avg_speed,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25,
        COUNT(*) AS records
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").show(24, truncate=False)

+-----------+---------+--------+-------+
|hour_of_day|avg_speed|avg_pm25|records|
+-----------+---------+--------+-------+
|22         |21.81    |12.47   |39250  |
|23         |25.06    |13.26   |805815 |
+-----------+---------+--------+-------+



In [20]:
# Wind speed as confounder - Does wind dilute the PM2.5 readings and reduce the correlation with traffic speed?
spark.sql("""
    SELECT
        CASE
            WHEN weather_wind_speed_mph < 5  THEN 'Calm (<5 mph)'
            WHEN weather_wind_speed_mph < 15 THEN 'Breezy (5-15 mph)'
            ELSE 'Windy (15+ mph)'
        END AS wind_category,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25,
        ROUND(AVG(traffic_speed), 2) AS avg_speed,
        COUNT(*) AS records
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL AND weather_wind_speed_mph IS NOT NULL
    GROUP BY 1
    ORDER BY avg_pm25 DESC
""").show(truncate=False)

+-----------------+--------+---------+-------+
|wind_category    |avg_pm25|avg_speed|records|
+-----------------+--------+---------+-------+
|Breezy (5-15 mph)|13.22   |24.91    |845065 |
+-----------------+--------+---------+-------+



In [21]:
# Top 10 most polluted H3 cells with low traffic speed
spark.sql("""
    SELECT
        h3_index,
        traffic_borough,
        ROUND(AVG(traffic_speed), 2) AS avg_speed,
        ROUND(AVG(aq_pm25_ugm3), 2) AS avg_pm25,
        COUNT(*) AS records,
        CAST(MAX(is_near_truck_route) AS STRING) AS on_truck_route
    FROM local.db.enriched_traffic
    WHERE aq_pm25_ugm3 IS NOT NULL AND traffic_speed < 15
    GROUP BY h3_index, traffic_borough
    ORDER BY avg_pm25 DESC
    LIMIT 10
""").show(truncate=False)

+---------------+---------------+---------+--------+-------+--------------+
|h3_index       |traffic_borough|avg_speed|avg_pm25|records|on_truck_route|
+---------------+---------------+---------+--------+-------+--------------+
|872a10623ffffff|Staten Island  |1.72     |20.45   |15987  |true          |
|872a100eeffffff|Queens         |11.12    |20.16   |4444   |true          |
|872a10729ffffff|Brooklyn       |12.87    |18.75   |32334  |true          |
|872a10729ffffff|Manhattan      |12.08    |16.81   |2880   |true          |
|872a100d0ffffff|Queens         |14.29    |15.55   |270    |true          |
|872a100abffffff|Bronx          |0.0      |14.27   |5389   |true          |
|872a1072dffffff|Manhattan      |8.17     |11.18   |62808  |false         |
|872a100d2ffffff|Manhattan      |9.66     |10.17   |26945  |false         |
|872a1000bffffff|Queens         |9.41     |6.35    |2190   |true          |
|872a1008bffffff|Manhattan      |0.0      |5.94    |64668  |true          |
+-----------

In [22]:
# Pipeline health summary card (good opening slide query)
spark.sql("""
    SELECT
        COUNT(*)                                                        AS total_enriched_records,
        COUNT(DISTINCT traffic_borough)                                 AS boroughs_covered,
        COUNT(DISTINCT h3_index)                                        AS unique_h3_cells,
        ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS aq_join_pct,
        ROUND(100.0 * SUM(CASE WHEN weather_temperature IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS weather_join_pct,
        MIN(traffic_event_ts)                                           AS pipeline_start,
        MAX(traffic_event_ts)                                           AS pipeline_latest
    FROM local.db.enriched_traffic
""").show(truncate=False)

+----------------------+----------------+---------------+-----------+----------------+-------------------+-------------------+
|total_enriched_records|boroughs_covered|unique_h3_cells|aq_join_pct|weather_join_pct|pipeline_start     |pipeline_latest    |
+----------------------+----------------+---------------+-----------+----------------+-------------------+-------------------+
|873659                |5               |49             |96.7       |99.8            |2026-05-06 21:18:00|2026-05-07 02:08:12|
+----------------------+----------------+---------------+-----------+----------------+-------------------+-------------------+

